In [5]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.Collecting xgboost
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 837.5 kB/s eta 0:02:01
   ---------------------------------------- 0.5/101.7 MB 837.5 kB/s eta 0:02:01
   ---------------------------------------- 0.8/101.7 MB 684.4 kB/s eta 0:02:28
   ---------------------------------------- 0.8/101.7 MB 684.4 kB/s eta 0:02:28
   ---------------------------------------- 1.0/101.7 MB 699.0 kB/s eta 0:02:24
   ---------------------------------------- 1.0/101.7 MB 699.0 kB/s eta 0:02:24
    --------------------------------------- 1.3/101.7 MB 691.7 kB/s eta 0:02:26
    --------------------------------------- 1.3/101.7 MB 691.7 kB/s eta 


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

from xgboost import XGBClassifier

In [7]:
df = pd.read_csv("train_test_network.csv")
df.head()

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [8]:
df = df.sample(30000, random_state=42)

In [9]:
X = df.drop(["label","type"],axis=1)
y = df["label"]

In [10]:
le = LabelEncoder()
for col in X.columns:
    if X[col].dtype=="object":
        X[col]=le.fit_transform(X[col].astype(str))

In [11]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30,random_state=42)

In [12]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [13]:
xgb = XGBClassifier(n_estimators=200,max_depth=8,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,random_state=42)

In [33]:
start=time.time()
xgb.fit(X_train,y_train)
end=time.time()
print("Training Time:",end-start)

Training Time: 0.48699235916137695


In [15]:
start=time.time()
pred=xgb.predict(X_test)
end=time.time()
print("Prediction Time:",end-start)

Prediction Time: 0.010896444320678711


In [16]:
print("XGBoost Accuracy:",accuracy_score(y_test,pred))

XGBoost Accuracy: 0.9997777777777778


In [17]:
from sklearn.ensemble import (RandomForestClassifier,ExtraTreesClassifier,GradientBoostingClassifier,StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report
import time

In [21]:
base_models=[
('rf',RandomForestClassifier(n_estimators=100,random_state=42)),
('et',ExtraTreesClassifier(n_estimators=100,random_state=42)),
('gb',GradientBoostingClassifier(random_state=42))]

In [22]:
meta_model=LogisticRegression()

In [23]:
hybrid_model=StackingClassifier(estimators=base_models,final_estimator=meta_model,passthrough=False)

In [31]:
start=time.time()
hybrid_model.fit(X_train,y_train)
end=time.time()
print("Training Time:",end-start)

Training Time: 63.27422475814819


In [28]:
preds = hybrid_model.predict(X_test)

In [29]:
from sklearn.metrics import accuracy_score
print("Hybrid Accuracy:",accuracy_score(y_test,preds))

Hybrid Accuracy: 0.9997777777777778
